# Drug–Target Interaction Prediction

## Notebook 01: Data Acquisition

### Objective
This notebook focuses on acquiring experimentally measured drug–target
interaction data from the ChEMBL database.

### Biological Question
Can experimentally measured drug–target interactions be used to build
a dataset for machine-learning-based interaction prediction?

### Data Source
- ChEMBL
- Human protein targets
- Small-molecule compounds
- Experimentally measured bioactivity data

### Workflow
1. Initialize ChEMBL connection
2. Retrieve human single-protein targets
3. Inspect target information
4. Select a representative protein target
5. Retrieve binding activity data
6. Inspect activity measurements
7. Save raw data

In [1]:
from chembl_webresource_client.new_client import new_client
import pandas as pd
import os

print("ChEMBL connection initialized")

d:\New_folder_3\Anaconda\Lib\site-packages\chembl_webresource_client\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


ChEMBL connection initialized


## 2. Retrieve Human Protein Targets

### Objective
Retrieve biological targets that satisfy two conditions:

- Target type = Single Protein
- Organism = Homo sapiens

### Why?
Our project focuses on drug–protein interactions relevant to human
biology. Restricting the initial dataset to human single-protein targets
keeps the project focused and reduces unnecessary complexity.

### Output
A dataframe containing information about human protein targets available
in ChEMBL.

In [3]:
target = new_client.target

targets = target.filter(
    target_type="SINGLE PROTEIN",
    organism="Homo sapiens"
)

targets_df = pd.DataFrame(targets)

print("Number of targets retrieved:", len(targets_df))
targets_df.head()

Number of targets retrieved: 5869


,cross_references,organism,pref_name,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,Maltase-glucoamylase,False,CHEMBL2074,"[{'accession': 'O43451', 'component_descriptio...",SINGLE PROTEIN,9606
1,[],Homo sapiens,ATP-binding cassette sub-family C member 9,False,CHEMBL1971,"[{'accession': 'O60706', 'component_descriptio...",SINGLE PROTEIN,9606
2,[],Homo sapiens,"cGMP-specific 3',5'-cyclic phosphodiesterase",False,CHEMBL1827,"[{'accession': 'O76074', 'component_descriptio...",SINGLE PROTEIN,9606
3,[],Homo sapiens,Voltage-dependent T-type calcium channel subun...,False,CHEMBL1859,"[{'accession': 'O95180', 'component_descriptio...",SINGLE PROTEIN,9606
4,[],Homo sapiens,Dihydrofolate reductase,False,CHEMBL202,"[{'accession': 'P00374', 'component_descriptio...",SINGLE PROTEIN,9606


## 3. Inspect Target Dataset

Before selecting a target, we inspect the structure of the retrieved
dataset.

This helps us understand the available identifiers, target names,
organisms, and target classifications.

In [4]:
targets_df.columns.tolist()

['cross_references',
 'organism',
 'pref_name',
 'species_group_flag',
 'target_chembl_id',
 'target_components',
 'target_type',
 'tax_id']

### 3.1 Inspect Target Names and Types

The retrieved dataset contains human single-protein targets from ChEMBL.
We inspect the target identifiers, names, organisms, and target types to
verify that the filtering was successful.

In [5]:
targets_df[
    [
        "target_chembl_id",
        "pref_name",
        "organism",
        "target_type"
    ]
].head(20)

,target_chembl_id,pref_name,organism,target_type
0,CHEMBL2074,Maltase-glucoamylase,Homo sapiens,SINGLE PROTEIN
1,CHEMBL1971,ATP-binding cassette sub-family C member 9,Homo sapiens,SINGLE PROTEIN
2,CHEMBL1827,"cGMP-specific 3',5'-cyclic phosphodiesterase",Homo sapiens,SINGLE PROTEIN
3,CHEMBL1859,Voltage-dependent T-type calcium channel subun...,Homo sapiens,SINGLE PROTEIN
4,CHEMBL202,Dihydrofolate reductase,Homo sapiens,SINGLE PROTEIN
5,CHEMBL1862,Tyrosine-protein kinase ABL1,Homo sapiens,SINGLE PROTEIN
6,CHEMBL203,Epidermal growth factor receptor,Homo sapiens,SINGLE PROTEIN
7,CHEMBL204,Prothrombin,Homo sapiens,SINGLE PROTEIN
8,CHEMBL1801,Plasminogen,Homo sapiens,SINGLE PROTEIN
9,CHEMBL1910,Adenosine deaminase,Homo sapiens,SINGLE PROTEIN


### 3.2 Check Target Type Distribution

We check the distribution of target types to confirm that our filtering
has returned the expected type of biological target.

In [6]:
targets_df["target_type"].value_counts()

target_type
SINGLE PROTEIN    5869
Name: count, dtype: int64

### 3.3 Verify the Selected Target

Before retrieving activity data, we verify that CHEMBL231 is present
among the retrieved human single-protein targets.

In [7]:
targets_df[
    targets_df["target_chembl_id"] == "CHEMBL231"
][
    [
        "target_chembl_id",
        "pref_name",
        "organism",
        "target_type"
    ]
]

,target_chembl_id,pref_name,organism,target_type
100,CHEMBL231,Histamine H1 receptor,Homo sapiens,SINGLE PROTEIN


### Preview of Target Records

In [8]:
targets_df.head()

,cross_references,organism,pref_name,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,Maltase-glucoamylase,False,CHEMBL2074,"[{'accession': 'O43451', 'component_descriptio...",SINGLE PROTEIN,9606
1,[],Homo sapiens,ATP-binding cassette sub-family C member 9,False,CHEMBL1971,"[{'accession': 'O60706', 'component_descriptio...",SINGLE PROTEIN,9606
2,[],Homo sapiens,"cGMP-specific 3',5'-cyclic phosphodiesterase",False,CHEMBL1827,"[{'accession': 'O76074', 'component_descriptio...",SINGLE PROTEIN,9606
3,[],Homo sapiens,Voltage-dependent T-type calcium channel subun...,False,CHEMBL1859,"[{'accession': 'O95180', 'component_descriptio...",SINGLE PROTEIN,9606
4,[],Homo sapiens,Dihydrofolate reductase,False,CHEMBL202,"[{'accession': 'P00374', 'component_descriptio...",SINGLE PROTEIN,9606


### Important Columns

The main columns of interest are:

- `chembl_id` — ChEMBL identifier for the target
- `pref_name` — preferred target name
- `target_type` — classification of the target
- `organism` — organism associated with the target

These identifiers will later help us connect target information with
drug activity data and UniProt protein sequences.

## 4. Select a Representative Protein Target

### Objective
For the initial data-acquisition stage, we use one well-characterized
human protein target to understand the structure of ChEMBL activity data.

### Selected Target
ChEMBL ID: CHEMBL231

This target corresponds to the human Histamine H1 receptor.

### Why start with one target?
ChEMBL contains a very large amount of bioactivity data. Starting with
one target allows us to validate the data-acquisition workflow before
scaling it to multiple protein targets.

In [9]:
target_id = "CHEMBL231"

target_info = target.get(target_id)

target_info

{'cross_references': [],
 'organism': 'Homo sapiens',
 'pref_name': 'Histamine H1 receptor',
 'species_group_flag': False,
 'target_chembl_id': 'CHEMBL231',
 'target_components': [{'accession': 'P35367',
   'component_description': 'Histamine H1 receptor',
   'component_id': 121,
   'component_type': 'PROTEIN',
   'relationship': 'SINGLE PROTEIN',
   'target_component_synonyms': [{'component_synonym': 'H1-R',
     'syn_type': 'UNIPROT'},
    {'component_synonym': 'H1R', 'syn_type': 'UNIPROT'},
    {'component_synonym': 'HH1R', 'syn_type': 'UNIPROT'},
    {'component_synonym': 'Histamine H1 receptor', 'syn_type': 'UNIPROT'},
    {'component_synonym': 'HRH1', 'syn_type': 'GENE_SYMBOL'}],
   'target_component_xrefs': [{'xref_id': 'P35367',
     'xref_name': None,
     'xref_src_db': 'AlphaFoldDB'},
    {'xref_id': 'P35367', 'xref_name': None, 'xref_src_db': 'ExpressionAtlas'},
    {'xref_id': 'GO:0005829',
     'xref_name': 'cytosol',
     'xref_src_db': 'GoComponent'},
    {'xref_id': 'G

## 5. Retrieve Binding Activity Data

### Objective
Retrieve experimentally measured binding activity records associated
with the selected protein target.

### Biological Meaning

A drug or small molecule can bind to a protein target with different
levels of affinity.

Common measurements include:

- Ki — inhibition constant
- Kd — dissociation constant
- IC50 — concentration producing 50% inhibition

These measurements provide quantitative information about
drug–target interactions.

### Filtering
We initially focus on binding assays using:

`assay_type = "B"`

This gives us a manageable and biologically meaningful starting point
for constructing the interaction dataset.

In [12]:
activity = new_client.activity

activities = activity.filter(
    target_chembl_id=target_id,
    assay_type="B"
)

activity_df = pd.DataFrame(activities)

print("Number of activity records:", len(activity_df))

Number of activity records: 6816


## 6. Inspect Activity Data

The activity dataset contains experimental measurements describing
interactions between compounds and the selected protein target.

Before performing any preprocessing, we inspect the available columns
and sample records.

In [14]:
activity_df.columns.tolist()

['action_type',
 'activity_comment',
 'activity_id',
 'activity_properties',
 'assay_chembl_id',
 'assay_description',
 'assay_type',
 'assay_variant_accession',
 'assay_variant_mutation',
 'bao_endpoint',
 'bao_format',
 'bao_label',
 'canonical_smiles',
 'data_validity_comment',
 'data_validity_description',
 'document_chembl_id',
 'document_journal',
 'document_year',
 'ligand_efficiency',
 'modality',
 'molecule_chembl_id',
 'molecule_pref_name',
 'parent_molecule_chembl_id',
 'pchembl_value',
 'potential_duplicate',
 'qudt_units',
 'record_id',
 'relation',
 'src_id',
 'standard_flag',
 'standard_relation',
 'standard_text_value',
 'standard_type',
 'standard_units',
 'standard_upper_value',
 'standard_value',
 'target_chembl_id',
 'target_organism',
 'target_pref_name',
 'target_tax_id',
 'text_value',
 'toid',
 'type',
 'units',
 'uo_units',
 'upper_value',
 'value']

### Preview of Activity Records

In [15]:
activity_df.head()

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,72509,[],CHEMBL694561,Inhibition of [3H]- doxepine binding to histam...,B,None,None,BAO_0000201,...,Homo sapiens,Histamine H1 receptor,9606,None,None,Inhibition,%,UO_0000187,None,22.0
1,None,None,85576,[],CHEMBL694562,Inhibition of [3H]- doxepine binding to histam...,B,None,None,BAO_0000201,...,Homo sapiens,Histamine H1 receptor,9606,None,None,Inhibition,%,UO_0000187,None,0.0
2,None,None,100483,[],CHEMBL694561,Inhibition of [3H]- doxepine binding to histam...,B,None,None,BAO_0000201,...,Homo sapiens,Histamine H1 receptor,9606,None,None,Inhibition,%,UO_0000187,None,30.0
3,None,None,110076,[],CHEMBL691619,In vitro binding affinity of compound against ...,B,None,None,BAO_0000192,...,Homo sapiens,Histamine H1 receptor,9606,None,None,Ki,nM,UO_0000065,None,6.0
4,None,None,116755,[],CHEMBL691619,In vitro binding affinity of compound against ...,B,None,None,BAO_0000192,...,Homo sapiens,Histamine H1 receptor,9606,None,None,Ki,nM,UO_0000065,None,7.0


### Key Activity Variables

The most important variables for the next stages are:

- `molecule_chembl_id` — identifier of the compound
- `target_chembl_id` — identifier of the protein target
- `standard_type` — type of biological measurement
- `standard_value` — measured activity value
- `standard_units` — units of measurement
- `pchembl_value` — standardized activity measure when available

These variables will later be used to clean the dataset and define
interaction labels.

In [16]:
activity_df[
    [
        "molecule_chembl_id",
        "target_chembl_id",
        "standard_type",
        "standard_value",
        "standard_units",
        "pchembl_value"
    ]
].head(10)

,molecule_chembl_id,target_chembl_id,standard_type,standard_value,standard_units,pchembl_value
0,CHEMBL113310,CHEMBL231,Inhibition,22.0,%,None
1,CHEMBL112203,CHEMBL231,Inhibition,0.0,%,None
2,CHEMBL112710,CHEMBL231,Inhibition,30.0,%,None
3,CHEMBL100454,CHEMBL231,Ki,6.0,nM,8.22
4,CHEMBL715,CHEMBL231,Ki,7.0,nM,8.15
5,CHEMBL1125,CHEMBL231,Ki,22.0,nM,7.66
6,CHEMBL440512,CHEMBL231,Ki,87.0,nM,7.06
7,CHEMBL156071,CHEMBL231,Ki,2511886431509582.0,nM,None
8,CHEMBL156071,CHEMBL231,IC50,1258.93,nM,5.90
9,CHEMBL156071,CHEMBL231,Alpha,0.84,None,None


###  Distribution of Bioactivity Measurements

ChEMBL may contain multiple types of experimental measurements for the
same target.

We inspect the distribution of `standard_type` before deciding which
measurement type should be used for our machine-learning dataset.

In [17]:
activity_df["standard_type"].value_counts()

standard_type
Ki                                                     2502
AC50                                                   1257
IC50                                                   1141
Inhibition                                              974
Activity                                                199
Kd                                                      193
EC50                                                     67
Intrinsic activity                                       62
Alpha                                                    41
% Ctrl                                                   39
k_off                                                    38
kon                                                      38
Kb                                                       35
pKi                                                      30
pEC50                                                    24
Activity_index                                           24
Retention_time            

###  Distribution of Bioactivity Units

We inspect the units used for the reported activity values.

Consistent units are required before numerical activity values can be
compared or used to construct interaction labels.

In [18]:
activity_df["standard_units"].value_counts()

standard_units
nM            3432
%             1149
s-1             38
/min            26
10^6/M/min      24
min             22
hr               5
M                2
Name: count, dtype: int64

### 6.5 Missing Values in Key Variables

We check whether important compound and bioactivity fields contain
missing values.

Missing values will be handled during the data-cleaning stage rather
than being removed without inspection.

In [19]:
activity_df[
    [
        "molecule_chembl_id",
        "target_chembl_id",
        "standard_type",
        "standard_value",
        "standard_units",
        "pchembl_value"
    ]
].isnull().sum()

molecule_chembl_id       0
target_chembl_id         0
standard_type            0
standard_value        2109
standard_units        2118
pchembl_value         4804
dtype: int64

## 7. Save Raw Activity Data

### Objective
Save the retrieved ChEMBL activity data without modifying it.

### Why keep raw data?

The raw dataset should always be preserved separately from processed
data. This allows us to:

- reproduce the analysis
- return to the original records
- compare raw and processed datasets
- troubleshoot preprocessing decisions

The raw dataset will be stored in:

`data/raw/h1_receptor_activity.csv`

In [20]:
os.makedirs("../data/raw", exist_ok=True)

activity_df.to_csv(
    "../data/raw/h1_receptor_activity.csv",
    index=False
)

print("Raw activity data saved successfully.")

Raw activity data saved successfully.


## 8. Data Acquisition Summary

### What was completed?

- Connected to the ChEMBL database using its Python client.
- Retrieved human single-protein targets.
- Selected CHEMBL231 (Histamine H1 receptor) as the initial target.
- Retrieved experimentally measured binding activity records.
- Inspected compound, target, and activity information.
- Saved the raw activity dataset for further processing.

### Current Dataset

The dataset contains experimentally measured interactions between
small-molecule compounds and the selected human protein target.

### Next Step

The raw data cannot be directly used for machine learning.

The next notebook will focus on:

**Raw Activity Data**
→ Data Cleaning
→ Activity Standardization
→ Duplicate Handling
→ Missing Values
→ Interaction Label Construction
→ Processed Dataset

### Important Note

Interaction labels will not be assigned arbitrarily. The definition
of an active/inactive interaction will be based on the selected
bioactivity measurement and appropriate biological criteria.